# Semantic Search using FAISS + Embeddings

**FAISS: Facebook AI Similarity Search**
```
FAISS finds similarity using:
- L2 Distance (Euclidean distance)
- Cosine similarity (via normalization)
```

## Objective (Explain to students)

1) Convert text → embeddings

2) Store in vector database using FAISS

3) Perform semantic search

## The Mathematical Trick: $L_2$ Distance vs. Cosine Similarity
FAISS is optimized to find the nearest vectors using Squared Euclidean Distance ($L_2$).
To make FAISS calculate Cosine Similarity using $L_2$, you must first normalize all vectors to a length of 1. When vectors have a length of 1 ($\vert{}\vert{}A\vert{}\vert{} = 1$ and $\vert{}\vert{}B\vert{}\vert{} = 1$), minimizing the Euclidean distance is mathematically identical to maximizing Cosine Similarity.

## Markdown Equations for Jupyter
You can copy and paste the raw text blocks below directly into a Jupyter Markdown cell.
## 1. Standard Cosine Similarity Formula
This is the standard definition of Cosine Similarity between two document vectors $A$ and $B$:

$$\text{Cosine Similarity}(A, B) = \frac{A \cdot B}{\|A\| \|B\|}$$

## 2. Vector Normalization
To bridge the gap to FAISS, we normalize every document vector so that its length (magnitude) equals 1: [5] 

$$\|A\| = 1 \quad \text{and} \quad \|B\| = 1 \quad \implies \quad \text{Cosine Similarity}(A, B) = A \cdot B$$

## 3. The Squared $L_2$ Distance Relationship
When we expand the algebraic equation for the Squared Euclidean ($L_2$) Distance between these two normalized vectors, we get:

$$\|A - B\|^2 = \|A\|^2 + \|B\|^2 - 2(A \cdot B)$$

## 4. The Final Connection
Since $\Vert{}A\Vert{}^2 = 1$ and $\Vert{}B\Vert{}^2 = 1$, we substitute those numbers into the equation to see exactly how FAISS finds the best match:

$$\|A - B\|^2 = 1 + 1 - 2(\text{Cosine Similarity})$$

$$\|A - B\|^2 = 2 - 2(\text{Cosine Similarity})$$

## The Logic

* Perfect Match: If two documents are identical, their Cosine Similarity is $1$. The FAISS $L_2$ distance becomes $2 - 2(1) = \mathbf{0}$.
* Opposite Meanings: If two documents are completely opposite, their Cosine Similarity is $-1$. The FAISS $L_2$ distance becomes $2 - 2(-1) = \mathbf{4}$.

Summary for your slides: FAISS searches for the smallest possible $L_2$ distance, which automatically extracts the documents with the highest Cosine Similarity.


In [2]:
# Install

!python -m pip install faiss-cpu==1.15.0 sentence-transformers==6.0.1

  Using cached setuptools-84.0.0-py3-none-any.whl.metadata (6.6 kB)
   ---------------------------------------- 0.0/16.2 MB ? eta -:--:--
   ---------------------------------------- 0.2/16.2 MB 5.3 MB/s eta 0:00:04
   ---------------------------------------- 0.2/16.2 MB 5.3 MB/s eta 0:00:04
   ---------------------------------------- 0.2/16.2 MB 5.3 MB/s eta 0:00:04
   ---------------------------------------- 0.2/16.2 MB 5.3 MB/s eta 0:00:04
   ---------------------------------------- 0.2/16.2 MB 5.3 MB/s eta 0:00:04
   ---------------------------------------- 0.2/16.2 MB 5.3 MB/s eta 0:00:04
   - -------------------------------------- 0.4/16.2 MB 1.4 MB/s eta 0:00:12
   -- ------------------------------------- 0.9/16.2 MB 2.5 MB/s eta 0:00:07
   --- ------------------------------------ 1.6/16.2 MB 4.2 MB/s eta 0:00:04
   ----- ---------------------------------- 2.1/16.2 MB 4.9 MB/s eta 0:00:03
   ----- ---------------------------------- 2.2/16.2 MB 5.2 MB/s eta 0:00:03
   ----- ------


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import importlib.metadata

# List the distribution package names
packages = ["faiss-cpu", "sentence-transformers"]

for package in packages:
    try:
        version = importlib.metadata.version(package)
        print(f"{package} version: {version}")
    except importlib.metadata.PackageNotFoundError:
        print(f"{package} is not installed in this environment.")


# faiss-cpu version: 1.15.0
# sentence-transformers version: 6.0.1

faiss-cpu version: 1.15.0
sentence-transformers version: 6.0.1


In [4]:
# Import Libraries: Took a 4-5 minutes

import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

# Load Embedding Model

In [5]:
# Pretrained embedding model: Took 4 minutes

model = SentenceTransformer('all-MiniLM-L6-v2') # works. Total parameters ≈ 22.7 Million
# model = SentenceTransformer('paraphrase-MiniLM-L3-v2') # works: Total parameters ≈ 17.39 Million
# model = SentenceTransformer('BAAI/bge-small-en') # works. Total parameters ≈ 33.36 Million

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

C:\Users\ash32\Desktop\education\play_langchain_langgraph\.venv\Lib\site-packages\huggingface_hub\file_download.py:142: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ash32\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

- This model converts text → vectors
- Each sentence becomes a numeric representation

# Lets check the model size

In [6]:
params = sum(p.numel() for p in model.parameters())

# Each parameter ~4 bytes (float32)
size_mb = params * 4 / (1024 * 1024)

print(f"Total parameters ≈ {params/1e6:.2f} Million")
print(f"Model size ≈ {size_mb:.2f} MB")

Total parameters ≈ 22.71 Million
Model size ≈ 86.64 MB


# Where is the model stored locally

In [7]:
import os
import pathlib
from sentence_transformers import SentenceTransformer

model_path = model._first_module().auto_model.config._name_or_path
print("model_path:", model_path)

# If downloaded locally, check cache folder
cache_dir = pathlib.Path.home() / ".cache" / "huggingface"
print("cache_dir:", cache_dir)

model_path: sentence-transformers/all-MiniLM-L6-v2
cache_dir: C:\Users\ash32\.cache\huggingface


# Create Sample Dataset

In [8]:
documents = [
    "Machine learning is a subset of AI",
    "Deep learning uses neural networks",
    "Python is used for data science",
    "Cars and vehicles are transportation",
    "Artificial intelligence is transforming industries",
    "Football is a popular sport",
    "Data engineering involves pipelines",
    "Big data tools include Hadoop and Spark"
]

# Convert Text → Embeddings

In [9]:
embeddings = model.encode(documents)
print("Shape of embeddings:", embeddings.shape)

Shape of embeddings: (8, 384)


- Each sentence → vector of numbers
- Shape = (num_docs, vector_size)

In [13]:
# Lets prints 1st document and its embeddings
print("document:", documents[0])

print("--------------------\n")

print("Embeddings:", embeddings[0])

document: Machine learning is a subset of AI
--------------------

Embeddings: [-4.87040840e-02 -1.66195333e-02  6.68975338e-02  3.49244811e-02
  6.72051534e-02 -1.57562792e-02  3.64907011e-02 -1.68502424e-02
 -2.90400870e-02 -2.84839608e-03 -7.59700090e-02 -1.85295586e-02
  5.06864674e-02 -6.54848143e-02  6.47153286e-03  2.77101751e-02
 -2.04247851e-02 -1.30060045e-02 -3.15499939e-02 -6.55897558e-02
 -1.98025294e-02 -4.39425325e-03 -5.25192060e-02 -1.06857987e-02
 -1.10507607e-02  7.10865632e-02  1.79339908e-02  4.04883064e-02
 -3.67941782e-02  5.33591360e-02  2.52874065e-02  1.64846424e-03
  9.88384057e-03 -1.60371065e-02 -1.14632854e-02  3.31828222e-02
 -3.52318995e-02  4.52054255e-02  6.44559562e-02  2.56205462e-02
 -1.16182417e-02 -2.91470774e-02  6.77248975e-03 -2.35240031e-02
  5.28734922e-02  1.14719562e-01 -7.95150474e-02 -5.55547848e-02
 -9.10266675e-03 -2.21333327e-03 -1.29922315e-01 -4.30228896e-02
 -1.68929808e-02  3.67006985e-03 -3.11332364e-02  2.30015144e-02
  4.5493077

# Create FAISS Index

In [15]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)  # L2 distance: Euclidean distance (similarity measure)

index.add(embeddings)

print("Number of vectors in DB:", index.ntotal)

Number of vectors in DB: 8


- FAISS stores vectors
- Enables fast similarity search

# Perform Semantic Search

In [17]:
query = "AI is changing the world"

query_embedding = model.encode([query])

k = 3  # top results
distances, indices = index.search(query_embedding, k)

print("Top results:")
for i in indices[0]:
    print(documents[i])

Top results:
Artificial intelligence is transforming industries
Machine learning is a subset of AI
Deep learning uses neural networks


# Add more realistic data

In [18]:
documents.extend([
    "Chatbots are powered by large language models",
    "Healthcare is improved by AI diagnostics. It is bringing positive change in the world",
    "Self-driving cars use machine learning"
])

embeddings = model.encode(documents)

print("Shape of embeddings:", embeddings.shape)

dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)  # L2 distance (similarity measure)
index.add(embeddings)

print("Number of vectors in DB:", index.ntotal)

Shape of embeddings: (11, 384)
Number of vectors in DB: 11


In [19]:
query = "AI is changing the world"

query_embedding = model.encode([query])

k = 3  # top results
distances, indices = index.search(query_embedding, k)

print("Top results:")
for i in indices[0]:
    print(documents[i])

Top results:
Artificial intelligence is transforming industries
Healthcare is improved by AI diagnostics. It is bringing positive change in the world
Machine learning is a subset of AI
